# Pályázati felhívás kockázati elemző – futtatható demo notebook

A projekt célja egy olyan MI-alapú támogatási felhívás elemző pipeline kialakítása, amely képes a pályázati PDF dokumentumok automatikus feldolgozására, strukturált mezőkinyerésre, kategorizálásra, intent felismerésre és kockázati értékelésre. A rendszer HuggingFace pipeline-okat, természetes nyelvfeldolgozási módszereket és lokálisan futtatott nagy nyelvi modellt (Mistral/Ollama) alkalmaz.
Ez a notebook bemutatja a beadandó pipeline fő lépéseit: PDF szövegkinyerés, ZSC kategorizálás, intent felismerés, LLM összefoglaló, kockázati pontszámítás és adatbázisba mentés.

In [10]:
import sys
!{sys.executable} -m pip install pdfplumber
from pathlib import Path
import sys
import os

BASE_DIR = Path(r"C:\Users\Kitti\OneDrive\Desktop\palyazat_fixed")

sys.path.insert(0, str(BASE_DIR))
os.chdir(BASE_DIR)

print("Projekt könyvtár:", BASE_DIR)
print("Aktuális könyvtár:", Path.cwd())
print("App mappa létezik:", (BASE_DIR / "app").exists())
print("PDF mappa létezik:", (BASE_DIR / "data" / "pdfs").exists())
print("PDF-ek száma:", len(list((BASE_DIR / "data" / "pdfs").glob("*.pdf"))))

Projekt könyvtár: C:\Users\Kitti\OneDrive\Desktop\palyazat_fixed
Aktuális könyvtár: C:\Users\Kitti\OneDrive\Desktop\palyazat_fixed
App mappa létezik: True
PDF mappa létezik: True
PDF-ek száma: 22


In [11]:
from pathlib import Path
import sys
import os

BASE_DIR = Path(r"C:\Users\Kitti\OneDrive\Desktop\palyazat_fixed")

sys.path.insert(0, str(BASE_DIR))
os.chdir(BASE_DIR)

print("Projekt könyvtár:", BASE_DIR)
print("Aktuális könyvtár:", Path.cwd())
print("App mappa létezik:", (BASE_DIR / "app").exists())
print("PDF mappa létezik:", (BASE_DIR / "data" / "pdfs").exists())
print("PDF-ek száma:", len(list((BASE_DIR / "data" / "pdfs").glob("*.pdf"))))

Projekt könyvtár: C:\Users\Kitti\OneDrive\Desktop\palyazat_fixed
Aktuális könyvtár: C:\Users\Kitti\OneDrive\Desktop\palyazat_fixed
App mappa létezik: True
PDF mappa létezik: True
PDF-ek száma: 22


## 1. PDF-ek listázása

In [12]:
pdf_dir = BASE_DIR / "data" / "pdfs"
pdf_files = sorted(pdf_dir.glob("*.pdf"))[:3]
pdf_files

[WindowsPath('C:/Users/Kitti/OneDrive/Desktop/palyazat_fixed/data/pdfs/ginop-plusz-121-felhivas.pdf'),
 WindowsPath('C:/Users/Kitti/OneDrive/Desktop/palyazat_fixed/data/pdfs/ginop-plusz-122-22-felhivas.pdf'),
 WindowsPath('C:/Users/Kitti/OneDrive/Desktop/palyazat_fixed/data/pdfs/ginop-plusz-123-felhivas.pdf')]

## 2. Egy PDF szövegének kinyerése és mezőkinyerés

In [13]:
from app.extractor import read_pdf_text, extract_fields
text = read_pdf_text(pdf_files[0])
fields = extract_fields(text)
fields

{'call_code': 'GINOP Plusz-1.2.1-21',
 'title': 'alkalmazkodását segítő fejlesztések támogatása',
 'advance_percent': 60,
 'own_fund_percent': None,
 'own_fund_required': 'igen',
 'consortium_allowed': 'nem',
 'support_type': 'vissza nem térítendő',
 'project_duration_months': 24,
 'max_support': 629300000,
 'min_support': 10000000,
 'total_budget_huf': 121100000000,
 'submission_start': None,
 'submission_end': None,
 'advance_max': None,
 'beneficiary_text': None,
 'activity_count': None,
 'project_count': None,
 'location_text': None}

## 3. ZSC kategorizálás

In [14]:
from app.zsc_classifier import classify_text
category = classify_text(text)
category

'kutatás-fejlesztés'

## 4. Intent recognizer

In [15]:
from app.intent_model import IntentRecognizer
intent_model = IntentRecognizer()
intent_model.predict("Milyen kockázatok vannak ebben a felhívásban?")

np.str_('risk_analysis')

## 5. Kockázati pontszámítás

In [16]:
from app.risk_model import compute_risk
risk = compute_risk(fields)
risk

(13,
 'magas',
 {'előleg': {'pont': 3,
   'megjegyzés': '60% – magas előleg, jelentős visszakövetelési kockázat'},
  'futamidő': {'pont': 2, 'megjegyzés': '24 hónap – közepes futamidő'},
  'max_támogatás': {'pont': 3, 'megjegyzés': '629 M Ft – nagy összeg'},
  'projektek_száma': {'pont': 1,
   'megjegyzés': 'ismeretlen – nem értékelhető (1 pont büntetés)'},
  'támogatás_típusa': {'pont': 3,
   'megjegyzés': 'vissza nem térítendő – nincs visszafizetési biztosíték, EU-szabálytalanság esetén kötelezettségszegési eljárás'},
  'önerő': {'pont': 1, 'megjegyzés': 'önerő szükséges, mértéke ismeretlen'},
  'konzorcium': {'pont': 0,
   'megjegyzés': 'konzorcium nem lehetséges – egy kedvezményezett, egyértelmű felelősség'}})

## 6. LLM összefoglaló – Mistral/Ollama
Ez a cella akkor fut jól, ha az Ollama és a Mistral modell elérhető.

In [17]:
from app.llm_summary import generate_stable_summary, is_ollama_available
is_ollama_available()
# summary = generate_stable_summary(text)
# print(summary[:2000])

(True, 'mistral:latest, tinyllama:latest')

## 7. Teljes pipeline futtatása
A parancs a projekt gyökeréből futtatható.

In [18]:
# !PDF_LIMIT=3 python -m app.ingest
# !python -m app.monitoring